# Week 4 Assignment – CIFAR-10 Image Classification

## ANN vs CNN: Comparing Architectures & Training Strategies

**Author:** Sahil Yadav  
**Internship:** Celebal Technologies – Data Science  
**Dataset:** [CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html) (via `tensorflow.keras.datasets`)  
**Reference Notebook:** CIFAR10_ANN_CNN_Learning_Project (Provided)  

---

### Objective
Build image classification models on the **CIFAR-10 dataset** using both an **Artificial Neural Network (ANN)** and a **Convolutional Neural Network (CNN)**, then compare their performance across different architectures and training strategies (dropout, batch normalization, data augmentation, learning rate scheduling).

### Approach
1. Load & explore the CIFAR-10 dataset
2. Preprocess – normalize pixel values, flatten for ANN
3. Build & train a baseline **ANN** model
4. Build & train a baseline **CNN** model
5. Compare accuracy, loss curves, and generalization
6. Upgrade with **training strategies** – data augmentation, LR scheduling, EarlyStopping
7. Final comparison table + conclusions


---
## 1. Install & Import Libraries


In [1]:
!pip install -q tensorflow matplotlib numpy pandas seaborn scikit-learn


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


TensorFlow version: 2.18.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


---
## 2. Load & Explore CIFAR-10

CIFAR-10 has **60 000** colour images of size **32×32×3** split across 10 classes:  
Airplane, Automobile, Bird, Cat, Deer, Dog, Frog, Horse, Ship, Truck.

- 50 000 training images  
- 10 000 test images


In [3]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

class_names = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

print("Training set  :", x_train.shape, y_train.shape)
print("Test set      :", x_test.shape, y_test.shape)
print("Pixel range   :", x_train.min(), "-", x_train.max())
print("Data type     :", x_train.dtype)


Training set  : (50000, 32, 32, 3) (50000, 1)
Test set      : (10000, 32, 32, 3) (10000, 1)
Pixel range   : 0 - 255
Data type     : uint8


### 2.1 Visualize Sample Images
Let's look at a few random training images to get a feel for the data.


In [4]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
indices = np.random.choice(len(x_train), 10, replace=False)
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[indices[i]])
    ax.set_title(class_names[y_train[indices[i]][0]], fontsize=11)
    ax.axis("off")
plt.suptitle("Random CIFAR-10 Training Samples", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


### 2.2 Class Distribution
Quick sanity check – each class should have roughly 5 000 samples.


In [5]:
unique, counts = np.unique(y_train, return_counts=True)
plt.figure(figsize=(10, 4))
sns.barplot(x=[class_names[i] for i in unique], y=counts, palette='viridis')
plt.title("Training Set – Class Distribution")
plt.ylabel("Count")
plt.xlabel("Class")
for i, v in enumerate(counts):
    plt.text(i, v + 50, str(v), ha='center', fontsize=9)
plt.tight_layout()
plt.show()
print("Classes are balanced ✓")


Classes are balanced ✓


---
## 3. Preprocessing

- **Normalize** pixels from 0-255 → 0-1 for stable gradient descent  
- **Flatten** images to 1-D vectors (3072,) for the ANN  
- Keep 3-D images (32,32,3) for the CNN


In [6]:
# normalize
x_train_norm = x_train.astype('float32') / 255.0
x_test_norm  = x_test.astype('float32') / 255.0

# flatten for ANN
x_train_flat = x_train_norm.reshape(len(x_train_norm), -1)  # (50000, 3072)
x_test_flat  = x_test_norm.reshape(len(x_test_norm), -1)    # (10000, 3072)

print("Normalized range:", x_train_norm.min(), "-", x_train_norm.max())
print("Flat shape      :", x_train_flat.shape)


Normalized range: 0.0 - 1.0
Flat shape      : (50000, 3072)


---
## 4. Part 1 – ANN (Fully Connected Network)

ANN treats each image as a **flat vector of 3072 values**. It cannot capture spatial relationships between neighbouring pixels, so we expect lower accuracy compared to CNN.

Architecture:
- Dense(512, relu) → Dropout(0.3) → Dense(256, relu) → Dense(128, relu) → Dense(10, softmax)


In [7]:
ann_model = models.Sequential([
    layers.Dense(512, activation='relu', input_shape=(3072,)),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax')
])

ann_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

ann_model.summary()


Model: "sequential"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 512)            │     1,573,376 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼─────────────────────

### 4.1 Train the ANN


In [8]:
ann_history = ann_model.fit(
    x_train_flat, y_train,
    epochs=15,
    validation_split=0.1,
    batch_size=64,
    verbose=1
)


Epoch 1/15
704/704 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.2842 - loss: 2.0147 - val_accuracy: 0.3694 - val_loss: 1.7652
Epoch 2/15
704/704 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.3911 - loss: 1.7321 - val_accuracy: 0.4121 - val_loss: 1.6614
Epoch 3/15
704/704 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.4187 - loss: 1.6542 - val_accuracy: 0.4298 - val_loss: 1.6132
Epoch 4/15
704/704 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.4361 - loss: 1.6118 - val_accuracy: 0.4419 - val_loss: 1.5823
Epoch 5/15
704/704 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.4493 - loss: 1.5827 - val_accuracy: 0.4487 - val_loss: 1.5602
Epoch 6/15
704/704 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.4571 - loss: 1.5612 - val_accuracy: 0.4520 - val_loss: 1.5478
Epoch 7/15
704/704 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.4622 - loss: 1.5441 - val_accuracy: 0.4564 - val_loss: 1.5351
Epoch 8/15
704/704 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.4680 - loss: 1.5297 - val_accuracy: 0.

### 4.2 Evaluate ANN on Test Set


In [9]:
ann_test_loss, ann_test_acc = ann_model.evaluate(x_test_flat, y_test, verbose=0)
print(f"ANN Test Loss    : {ann_test_loss:.4f}")
print(f"ANN Test Accuracy: {ann_test_acc:.4f}")


ANN Test Loss    : 1.5124
ANN Test Accuracy: 0.4681


### 4.3 ANN – Learning Curves


In [10]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(ann_history.history['accuracy'], label='Train Accuracy', marker='o', markersize=4)
ax1.plot(ann_history.history['val_accuracy'], label='Val Accuracy', marker='s', markersize=4)
ax1.set_title("ANN – Accuracy")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy")
ax1.legend()
ax1.grid(True)

ax2.plot(ann_history.history['loss'], label='Train Loss', marker='o', markersize=4)
ax2.plot(ann_history.history['val_loss'], label='Val Loss', marker='s', markersize=4)
ax2.set_title("ANN – Loss")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()


### 4.4 ANN – Confusion Matrix & Classification Report


In [11]:
ann_preds = ann_model.predict(x_test_flat, verbose=0)
ann_pred_labels = np.argmax(ann_preds, axis=1)

print("Classification Report (ANN):")
print(classification_report(y_test, ann_pred_labels, target_names=class_names))

cm_ann = confusion_matrix(y_test, ann_pred_labels)
plt.figure(figsize=(10, 8))
sns.heatmap(cm_ann, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title("ANN – Confusion Matrix")
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
plt.show()


Classification Report (ANN):
              precision    recall  f1-score   support

    airplane       0.54      0.53      0.54      1000
  automobile       0.56      0.56      0.56      1000
        bird       0.34      0.33      0.34      1000
         cat       0.31      0.29      0.30      1000
        deer       0.39      0.36      0.38      1000
         dog       0.40      0.38      0.39      1000
        frog       0.46      0.55      0.50      1000
       horse       0.51      0.52      0.52      1000
        ship       0.56      0.57      0.56      1000
       truck       0.52      0.57      0.54      1000

    accuracy                           0.47     10000
   macro avg       0.46      0.47      0.46     10000
weighted avg       0.46      0.47      0.46     10000



---
## 5. Part 2 – CNN (Convolutional Neural Network)

Unlike ANN, CNN preserves the **spatial structure** of images using:
- **Convolutional layers** – learn local spatial patterns (edges, textures)
- **Pooling layers** – downsample feature maps, reduce computation
- **Batch Normalization** – stabilize training, allow higher learning rates

Architecture:  
Conv2D(32) → BN → MaxPool → Conv2D(64) → BN → MaxPool → Conv2D(128) → BN → Flatten → Dense(256) → Dropout(0.4) → Dense(10)


In [12]:
cnn_model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
])

cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()


Model: "sequential_1"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 16, 16, 32)

### 5.1 Train the CNN


In [13]:
cnn_history = cnn_model.fit(
    x_train_norm, y_train,
    epochs=15,
    validation_split=0.1,
    batch_size=64,
    verbose=1
)


Epoch 1/15
704/704 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.4523 - loss: 1.5342 - val_accuracy: 0.5742 - val_loss: 1.2143
Epoch 2/15
704/704 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.6182 - loss: 1.0812 - val_accuracy: 0.6541 - val_loss: 0.9912
Epoch 3/15
704/704 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.6801 - loss: 0.9187 - val_accuracy: 0.7012 - val_loss: 0.8621
Epoch 4/15
704/704 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.7143 - loss: 0.8212 - val_accuracy: 0.7268 - val_loss: 0.7912
Epoch 5/15
704/704 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.7362 - loss: 0.7601 - val_accuracy: 0.7411 - val_loss: 0.7489
Epoch 6/15
704/704 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.7534 - loss: 0.7128 - val_accuracy: 0.7523 - val_loss: 0.7198
Epoch 7/15
704/704 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.7668 - loss: 0.6753 - val_accuracy: 0.7592 - val_loss: 0.6987
Epoch 8/15
704/704 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.7773 - loss: 0.6448 - val_accu

### 5.2 Evaluate CNN on Test Set


In [14]:
cnn_test_loss, cnn_test_acc = cnn_model.evaluate(x_test_norm, y_test, verbose=0)
print(f"CNN Test Loss    : {cnn_test_loss:.4f}")
print(f"CNN Test Accuracy: {cnn_test_acc:.4f}")


CNN Test Loss    : 0.6589
CNN Test Accuracy: 0.7823


### 5.3 CNN – Learning Curves


In [15]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(cnn_history.history['accuracy'], label='Train Accuracy', marker='o', markersize=4)
ax1.plot(cnn_history.history['val_accuracy'], label='Val Accuracy', marker='s', markersize=4)
ax1.set_title("CNN – Accuracy")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy")
ax1.legend()
ax1.grid(True)

ax2.plot(cnn_history.history['loss'], label='Train Loss', marker='o', markersize=4)
ax2.plot(cnn_history.history['val_loss'], label='Val Loss', marker='s', markersize=4)
ax2.set_title("CNN – Loss")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()


### 5.4 CNN – Confusion Matrix & Classification Report


In [16]:
cnn_preds = cnn_model.predict(x_test_norm, verbose=0)
cnn_pred_labels = np.argmax(cnn_preds, axis=1)

print("Classification Report (CNN):")
print(classification_report(y_test, cnn_pred_labels, target_names=class_names))

cm_cnn = confusion_matrix(y_test, cnn_pred_labels)
plt.figure(figsize=(10, 8))
sns.heatmap(cm_cnn, annot=True, fmt='d', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names)
plt.title("CNN – Confusion Matrix")
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
plt.show()


Classification Report (CNN):
              precision    recall  f1-score   support

    airplane       0.83      0.80      0.82      1000
  automobile       0.88      0.89      0.88      1000
        bird       0.69      0.67      0.68      1000
         cat       0.60      0.60      0.60      1000
        deer       0.74      0.76      0.75      1000
         dog       0.69      0.68      0.69      1000
        frog       0.83      0.86      0.85      1000
       horse       0.83      0.84      0.83      1000
        ship       0.87      0.88      0.87      1000
       truck       0.85      0.86      0.85      1000

    accuracy                           0.78     10000
   macro avg       0.78      0.78      0.78     10000
weighted avg       0.78      0.78      0.78     10000



---
## 6. ANN vs CNN – Direct Comparison

### 6.1 Validation Accuracy & Loss Curves


In [17]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
ax1.plot(ann_history.history['val_accuracy'], label='ANN Val Acc', marker='o', markersize=4, linestyle='--')
ax1.plot(cnn_history.history['val_accuracy'], label='CNN Val Acc', marker='s', markersize=4)
ax1.set_title("Validation Accuracy – ANN vs CNN")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy")
ax1.legend()
ax1.grid(True)

# Loss
ax2.plot(ann_history.history['val_loss'], label='ANN Val Loss', marker='o', markersize=4, linestyle='--')
ax2.plot(cnn_history.history['val_loss'], label='CNN Val Loss', marker='s', markersize=4)
ax2.set_title("Validation Loss – ANN vs CNN")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()


### 6.2 Summary Table


In [18]:
comparison = pd.DataFrame({
    "Model": ["ANN (Baseline)", "CNN (Baseline)"],
    "Test Accuracy": [ann_test_acc, cnn_test_acc],
    "Test Loss": [ann_test_loss, cnn_test_loss],
    "Total Params": [ann_model.count_params(), cnn_model.count_params()]
})
comparison["Test Accuracy"] = comparison["Test Accuracy"].map(lambda x: f"{x:.4f}")
comparison["Test Loss"]     = comparison["Test Loss"].map(lambda x: f"{x:.4f}")
comparison


            Model Test Accuracy Test Loss  Total Params
0  ANN (Baseline)        0.4681    1.5124       1738890
1  CNN (Baseline)        0.7823    0.6589        667434


---
## 7. Training Strategy Upgrades

Now let's push the CNN further with:
1. **Data Augmentation** – random flips, rotations, shifts to reduce overfitting
2. **Learning Rate Scheduling** – reduce LR when validation loss plateaus
3. **EarlyStopping** – stop training if no improvement

These are standard tricks used in practice to squeeze more performance out of the same architecture.


### 7.1 Data Augmentation Setup


In [19]:
datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1
)
datagen.fit(x_train_norm)

# visualize some augmented samples
fig, axes = plt.subplots(1, 5, figsize=(12, 3))
sample_img = x_train_norm[0:1]
for i, ax in enumerate(axes):
    aug_img = datagen.random_transform(sample_img[0])
    ax.imshow(aug_img)
    ax.axis('off')
    ax.set_title(f"Aug {i+1}")
plt.suptitle("Augmented Versions of One Image", fontsize=13)
plt.tight_layout()
plt.show()


### 7.2 Enhanced CNN with Augmentation + Callbacks


In [20]:
# same architecture but train with augmentation + callbacks
enhanced_cnn = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

enhanced_cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# callbacks
early_stop = callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True, verbose=1
)
lr_scheduler = callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1
)

print("Enhanced CNN compiled with Adam, EarlyStopping, ReduceLROnPlateau")


Enhanced CNN compiled with Adam, EarlyStopping, ReduceLROnPlateau


### 7.3 Train Enhanced CNN


In [21]:
# use validation split manually so we can use datagen
val_split = 5000
x_val = x_train_norm[-val_split:]
y_val = y_train[-val_split:]
x_train_aug = x_train_norm[:-val_split]
y_train_aug = y_train[:-val_split]

enh_history = enhanced_cnn.fit(
    datagen.flow(x_train_aug, y_train_aug, batch_size=64),
    epochs=30,
    validation_data=(x_val, y_val),
    callbacks=[early_stop, lr_scheduler],
    verbose=1
)


Epoch 1/30
704/704 ━━━━━━━━━━━━━━━━━━━━ 14s 20ms/step - accuracy: 0.3987 - loss: 1.6823 - val_accuracy: 0.5321 - val_loss: 1.3142
Epoch 2/30
704/704 ━━━━━━━━━━━━━━━━━━━━ 14s 20ms/step - accuracy: 0.5642 - loss: 1.2341 - val_accuracy: 0.6342 - val_loss: 1.0521
Epoch 3/30
704/704 ━━━━━━━━━━━━━━━━━━━━ 14s 20ms/step - accuracy: 0.6321 - loss: 1.0512 - val_accuracy: 0.6912 - val_loss: 0.8912
Epoch 4/30
704/704 ━━━━━━━━━━━━━━━━━━━━ 14s 20ms/step - accuracy: 0.6749 - loss: 0.9318 - val_accuracy: 0.7218 - val_loss: 0.8012
Epoch 5/30
704/704 ━━━━━━━━━━━━━━━━━━━━ 14s 20ms/step - accuracy: 0.7003 - loss: 0.8562 - val_accuracy: 0.7412 - val_loss: 0.7423
Epoch 6/30
704/704 ━━━━━━━━━━━━━━━━━━━━ 14s 20ms/step - accuracy: 0.7198 - loss: 0.8012 - val_accuracy: 0.7563 - val_loss: 0.7012
Epoch 7/30
704/704 ━━━━━━━━━━━━━━━━━━━━ 14s 20ms/step - accuracy: 0.7342 - loss: 0.7598 - val_accuracy: 0.7674 - val_loss: 0.6712
Epoch 8/30
704/704 ━━━━━━━━━━━━━━━━━━━━ 14s 20ms/step - accuracy: 0.7459 - loss: 0.7263 - 

### 7.4 Evaluate Enhanced CNN


In [22]:
enh_test_loss, enh_test_acc = enhanced_cnn.evaluate(x_test_norm, y_test, verbose=0)
print(f"Enhanced CNN Test Loss    : {enh_test_loss:.4f}")
print(f"Enhanced CNN Test Accuracy: {enh_test_acc:.4f}")


Enhanced CNN Test Loss    : 0.5712
Enhanced CNN Test Accuracy: 0.8134


### 7.5 Enhanced CNN – Learning Curves


In [23]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(enh_history.history['accuracy'], label='Train Accuracy', marker='o', markersize=3)
ax1.plot(enh_history.history['val_accuracy'], label='Val Accuracy', marker='s', markersize=3)
ax1.set_title("Enhanced CNN – Accuracy")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy")
ax1.legend()
ax1.grid(True)

ax2.plot(enh_history.history['loss'], label='Train Loss', marker='o', markersize=3)
ax2.plot(enh_history.history['val_loss'], label='Val Loss', marker='s', markersize=3)
ax2.set_title("Enhanced CNN – Loss")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()


In [24]:
enh_preds = enhanced_cnn.predict(x_test_norm, verbose=0)
enh_pred_labels = np.argmax(enh_preds, axis=1)

print("Classification Report (Enhanced CNN):")
print(classification_report(y_test, enh_pred_labels, target_names=class_names))

cm_enh = confusion_matrix(y_test, enh_pred_labels)
plt.figure(figsize=(10, 8))
sns.heatmap(cm_enh, annot=True, fmt='d', cmap='Oranges',
            xticklabels=class_names, yticklabels=class_names)
plt.title("Enhanced CNN – Confusion Matrix")
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
plt.show()


Classification Report (Enhanced CNN):
              precision    recall  f1-score   support

    airplane       0.86      0.83      0.84      1000
  automobile       0.91      0.92      0.91      1000
        bird       0.73      0.72      0.72      1000
         cat       0.65      0.64      0.65      1000
        deer       0.79      0.81      0.80      1000
         dog       0.73      0.72      0.73      1000
        frog       0.87      0.89      0.88      1000
       horse       0.86      0.87      0.87      1000
        ship       0.89      0.91      0.90      1000
       truck       0.88      0.89      0.88      1000

    accuracy                           0.82     10000
   macro avg       0.82      0.82      0.82     10000
weighted avg       0.82      0.82      0.82     10000



### 7.6 Sample Misclassified Images (Enhanced CNN)
Let's look at some images the enhanced CNN got wrong to understand failure cases.


In [25]:
# find misclassified samples
wrong_idx = np.where(enh_pred_labels != y_test.flatten())[0]
sample_wrong = np.random.choice(wrong_idx, 10, replace=False)

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, ax in enumerate(axes.flat):
    idx = sample_wrong[i]
    ax.imshow(x_test_norm[idx])
    ax.set_title(f"True: {class_names[y_test[idx][0]]}\nPred: {class_names[enh_pred_labels[idx]]}",
                 fontsize=9, color='red')
    ax.axis('off')
plt.suptitle("Misclassified Samples (Enhanced CNN)", fontsize=14)
plt.tight_layout()
plt.show()


---
## 8. Final Comparison – All Models


In [26]:
final_comparison = pd.DataFrame({
    "Model": ["ANN (Baseline)", "CNN (Baseline)", "CNN + Augmentation + LR Schedule"],
    "Test Accuracy": ["0.4681", "0.7823", "0.8134"],
    "Test Loss": ["1.5124", "0.6589", "0.5712"],
    "Key Features": [
        "Flat input, Dense layers only",
        "Conv + BN + MaxPool + Dropout",
        "Augmentation + ReduceLR + EarlyStopping"
    ]
})
print("=" * 85)
print("FINAL MODEL COMPARISON")
print("=" * 85)
print(final_comparison.to_string(index=False))
print("=" * 85)


FINAL MODEL COMPARISON
                          Model Test Accuracy Test Loss                                 Key Features
                 ANN (Baseline)        0.4681    1.5124                 Flat input, Dense layers only
                 CNN (Baseline)        0.7823    0.6589                 Conv + BN + MaxPool + Dropout
CNN + Augmentation + LR Schedule        0.8134    0.5712 Augmentation + ReduceLR + EarlyStopping


In [27]:
models_list = ["ANN", "CNN", "CNN+Aug"]
accs = [0.4681, 0.7823, 0.8134]
losses = [1.5124, 0.6589, 0.5712]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

colors = ['#e74c3c', '#3498db', '#2ecc71']

bars1 = ax1.bar(models_list, accs, color=colors, edgecolor='black', linewidth=0.8)
ax1.set_title("Test Accuracy Comparison", fontsize=13, fontweight='bold')
ax1.set_ylabel("Accuracy")
ax1.set_ylim(0, 1.0)
for bar, acc in zip(bars1, accs):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f"{acc:.2%}", ha='center', fontsize=11, fontweight='bold')

bars2 = ax2.bar(models_list, losses, color=colors, edgecolor='black', linewidth=0.8)
ax2.set_title("Test Loss Comparison", fontsize=13, fontweight='bold')
ax2.set_ylabel("Loss")
for bar, loss in zip(bars2, losses):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f"{loss:.4f}", ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()


---
## 9. Key Observations & Analysis

### Why CNN >> ANN for Image Classification?

| Aspect | ANN | CNN |
|--------|-----|-----|
| **Input handling** | Flattens 32×32×3 → 3072 vector, loses spatial info | Preserves 2D spatial structure |
| **Feature extraction** | Learns global patterns only | Learns local → global features hierarchically |
| **Parameter efficiency** | 1.7M params for ~47% accuracy | 667K params for ~78% accuracy |
| **Translation invariance** | None – pixel position matters | Yes – detects features anywhere in the image |
| **Overfitting tendency** | High (too many params, no spatial priors) | Lower (weight sharing, pooling) |

### Effect of Training Strategies

1. **Data Augmentation** – reduced the train-val gap (less overfitting) by exposing the model to shifted, flipped, rotated versions of training images
2. **ReduceLROnPlateau** – helped the optimizer converge more carefully near the minimum once validation loss stopped improving rapidly
3. **EarlyStopping** – prevented unnecessary epochs and restored the best model weights, saving time and avoiding overfitting
4. **Dropout (increased to 0.5)** – stronger regularization before the final dense layer helped generalization

### Hardest Classes
Looking at the confusion matrices, **cat** and **dog** are the hardest classes for all models. This makes sense because:
- Cats and dogs have similar body shapes at 32×32 resolution
- Both are four-legged animals with similar textures
- The low resolution makes fine-grained details (like whiskers vs snout) hard to distinguish


---
## 10. Conclusion

- **ANN is not suitable for image classification** – it treats every pixel independently and cannot learn spatial patterns. Test accuracy: **~47%**
- **CNN significantly outperforms ANN** because convolution layers preserve and exploit the 2D spatial structure of images. Test accuracy: **~78%**  
- **Training strategies push CNN further** – with data augmentation, learning rate scheduling, and early stopping, the enhanced CNN reached **~81%** accuracy
- The biggest jump was from ANN → CNN (47% → 78%), confirming that **architecture choice matters more than training tricks** for image tasks
- For even higher accuracy, one could try:
  - Deeper architectures (ResNet, VGG)
  - Transfer learning with pretrained ImageNet weights
  - More aggressive augmentation (CutOut, MixUp)
  - Longer training with cosine annealing schedule

**This assignment helped me understand the complete deep learning pipeline and why CNNs are the go-to architecture for computer vision tasks.**
